# Using ML to predict whether an NFL team will run or pass the ball

The NFL (National Football League) is the highest level professional American football league in the United States. In American football, the goal of teams is to move the ball up the field to the opponent's end zone and score a touchdown, which is worth 6 points. On each play, the offense (attacking team) can move the ball in 2 ways: passing (throwing it to a player further up the field) or rushing (running with the ball).

The optimal way for the defending team to approach the play depends massively on whether the offense passes or rushes. The ability to effectively predict what type of play the opponent's offense will run is therefore of great importance to teams. Luckily, it is not completely random. There is added situational context to every play that can lend itself to either a passing or rushing being preferred.

The offense always has 4 attempts to move the ball 10 yards up the field; once they do, the count resets and they again have 4 attempts for the next 10 yards. The attempt is called a "down". Hence, saying a team is on "2nd down and 8", means they are on their 2nd attempt out of 4 and have 8 yards to go. There are some general patterns for when a team will prioritise either a run or a pass. In general, the fewer attempts a team has and the further it needs to go, the more they lean towards passing the ball, since a throw can travel further in less time.

Teams also behave differently based on what the current score is. If a team is preserving a lead, they may play differently than if they are losing by a lot and chasing the game. Naturally, the time of the game also plays a role, since as the game nears its end, teams may start getting more desperate and take more risks. Other features include timeouts remaining for both teams. Timeouts give teams the ability to stop the clock, and are mainly used when teams are losing and the time is about to expire.

All the aforementioned features are what I will refer to as "game-context". All these features are known about a play the moment the previous play ends. There are then further features which I will refer to as "pre-snap" features, which relate to how the team on offense behaves right before the play starts. Specifically, I will look at whether the offense lines up in a "shotgun" formation (where the quarterback, the player who typically receives the snap and starts the play, stands several yards behind the center rather than directly behind them) and whether or not they form a "huddle" to discuss the play beforehand.

The nature of American football, being split into discrete plays that have a clear start and end, lends itself nicely to ML-based prediction.

This project has 3 main aims:
1. To examine the extent to which I can use machine-learning to predict whether a team will pass or rush based on situational context
2. To examine how much additional predictive power "pre-snap" features have when added to "game-context" features
3. To analyse what this comparison tells us about the nature of the models used

## Data Setup 
I begin by importing the libraries used for data retrieval, manipulation, modelling, and evaluation.

In [1]:
# NFL data
import nflreadpy as nfl

# Data manipulation
import pandas as pd

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Scaling for logistic regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

## Data and Feature Selection

I first load the 2025 NFL play-by-play data. The raw dataset contains hundreds of columns, so I retain only variables that describe the game state and offensive setup before the play begins.

The target variable is `play_type`, which I later restrict to run and pass plays.

The selected variables are:

- `week`: week of the NFL season
- `posteam`: team in possession of the ball
- `defteam`: defending team
- `down`: current down
- `ydstogo`: yards required for a first down
- `yardline_100`: distance in yards from the opponent's end zone
- `qtr`: current quarter
- `game_seconds_remaining`: seconds remaining in the game
- `home_timeouts_remaining`: home team's remaining timeouts
- `away_timeouts_remaining`: away team's remaining timeouts
- `score_differential`: offensive team's score minus defensive team's score
- `shotgun`: whether the offense is lined up in a shotgun formation
- `no_huddle`: whether the offense is operating without a huddle

In [2]:
pbp = nfl.load_pbp(seasons=2025).to_pandas()

pbp = pbp[[
    'week',
    'posteam',
    'defteam',
    'play_type',
    'down',
    'ydstogo',
    'yardline_100',
    'qtr',
    'game_seconds_remaining',
    'home_timeouts_remaining',
    'away_timeouts_remaining',
    'score_differential',
    'shotgun',
    'no_huddle'
]]

I restrict the dataset to only include run and pass plays. Other situations like field goals or punts are not relevant.

In [3]:
pbp = pbp[(pbp['play_type'] == ('run')) | (pbp['play_type'] == ('pass'))]

## Incorporating team playstyle into the model

There is one feature not provided by the data that I have to craft myself. This feature is rush tendency: for a given team, on what % of plays have they run the ball so far, looking at all preceding weeks in the current season. This allows the models to also consider team-specific tendencies when predicting what the play will be. To craft this feature, I first have to import week-by-week team stats, and create the `game_rush_tendency` variable, describing what % of plays a team ran the ball in a given game.

In [4]:
team = nfl.load_team_stats(seasons=2025, summary_level='week').to_pandas()

# 'attempts' refers to pass attempts, 'carries' to rush attempts
team['game_rush_tendency'] = team['carries'] / (team['carries'] + team['attempts'])

### Restricting the timespan of the dataset
The dataset features 22 weeks of the NFL season. The first 18 weeks are the regular season, the last 4 are the playoffs. I restrict the dataset to only feature the regular season, since the playoffs, with single-game elimination, may see a different style of play. Since the training period will be all regular season, testing on playoff games that are not represented in the training sample could lead to inaccurate evaluation statistics for the models.

Additionally, week 1 has to be excluded due to the nature of the `rush_tendency` variable. As stated, this variable in a given week is constructed using only data from previous weeks. For week 1, there are no previous weeks in the dataset. I considered using previous seasons' data; however, so much changes between seasons (coaching, personnel, playstyle) that this would likely be unrepresentative.

In [5]:
# Include only regular season
pbp = pbp[pbp['week'] < 19]

# Eliminate week 1 since no pre-week 1 data
pbp = pbp[pbp['week'] > 1]

### Creating the `rush_tendency` variable

The variable itself is created by, for each row of the play-by-play dataframe, examining the team dataframe with week-by-week data per team ONLY UP TO the previous week, then averaging the `game_rush_tendency` variable across all those previous games for that given team.

In [6]:
pbp['rush_tendency'] = pbp.apply(
    lambda row: team[team['week'] < row['week']].groupby('team')['game_rush_tendency'].mean()[row['posteam']],
    axis=1
)

I noticed some values of the `down` column were N/A. This is likely because those plays were 2-point conversions. This can happen (although rarely) after a team scores a touchdown and elects not to kick the ball for one extra point, but instead run a play for a potential 2 points. These plays are an exception, as even though teams still choose whether to pass or run the ball, there is only one attempt, so when it comes to prediction, these plays should be treated as their own situation, separate from normal plays.

In [7]:
# Filter out rows with down=N/A, presumably 2-point conversions
pbp = pbp[pbp['down'].notna()]

## Model Training

I chose the train-test split to be roughly 75-25, meaning models are trained on weeks 2 to 14, and then evaluated on weeks 15 to 18.

### Part 1: Game-context features only

First, the feature set is restricted to only game-context features, those that are known about the play immediately after the previous play ends. This means excluding the two pre-snap variables: `no_huddle` and `shotgun`.

In [8]:
train = pbp[pbp['week'] <= 14]
test = pbp[pbp['week'] > 14]

# Training datasets
X_train, y_train = train[['ydstogo', 'down', 'yardline_100', 
           'score_differential', 'game_seconds_remaining', 
           'home_timeouts_remaining', 'away_timeouts_remaining', 'qtr', 'rush_tendency']], train['play_type']

# Testing datasets
X_test, y_test = test[['ydstogo', 'down', 'yardline_100', 
           'score_differential', 'game_seconds_remaining', 
           'home_timeouts_remaining', 'away_timeouts_remaining', 'qtr', 'rush_tendency']], test['play_type']

### Baseline Accuracy

To give a baseline accuracy for model comparison, I use a majority-class baseline. This means that if a model were to simply predict the more common play (in this case pass) every time, it would be correct 55% of the time. 

In [9]:
print(y_test.value_counts(normalize=True))

play_type
pass    0.554641
run     0.445359
Name: proportion, dtype: float64


### Model 1: Logistic Regression

The first model I test is logistic regression. The main drawback of this model is its inability to recognise non-linear relationships in the data. One variable where this may be an issue is the `qtr` variable. After the 1st and 3rd quarters end, teams just switch sides and whichever team was on offense continues from the exact same situation. After the 2nd quarter ends, however, the game goes to halftime, after which the play resets. This adds a sense of urgency for whichever team is on offense, as it has to try to score before the quarter ends, which is not the case in the 1st and 3rd quarters. Naturally, 4th quarters also have more urgency as the game is over after them. This means that we would expect teams in the 2nd and 4th quarter to play with more urgency than in the 1st and 3rd, yet logistic regression will not be able to account for this non-linear relationship, and will assign `qtr` a single coefficient, assuming that any change in play tendency is linear as quarters increase.

In [10]:
clf_log = LogisticRegression(max_iter=1000)
clf_log.fit(X_train, y_train)

y_pred_log = clf_log.predict(X_test)

acc_log = accuracy_score(y_test, y_pred_log)
print(f"Overall accuracy: {acc_log}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_log)}")

pass_prob_log = clf_log.predict_proba(X_test)[:,0]
auc_log = roc_auc_score(y_test == 'pass', pass_prob_log)

pred_dist_log = pd.Series(y_pred_log).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_log.to_string()}\n")

print(f"AUC: {auc_log}")

Overall accuracy: 0.654640522875817

Classification Report:

              precision    recall  f1-score   support

        pass       0.67      0.75      0.71      4243
         run       0.63      0.54      0.58      3407

    accuracy                           0.65      7650
   macro avg       0.65      0.64      0.64      7650
weighted avg       0.65      0.65      0.65      7650

Predicted play type distribution:
pass    0.61817
run     0.38183

AUC: 0.7064102749458508


The logistic model gives an overall accuracy of 65.5%, already 10 percentage points above the 55.5% baseline. This shows that simple linear relationships based on game-context can already capture a substantial amount of signal. Precision is only slightly higher for 'pass' (67% vs 63%), while recall is much higher (75% vs 54%). This asymmetry reflects a bias towards passing: the model predicts 61.8% of plays as passes, whereas only 55.5% truly are, so it catches most passes but misses nearly half of all runs. The AUC of 0.706 means that a randomly chosen pass play is assigned a higher pass probability than a randomly chosen run play about 70.6% of the time, indicating moderate ability to separate the two play types.

### Model 2: Single Decision Tree (unrestricted depth)

The second model I test is a single decision tree with unrestricted depth. The main difference compared to logistic regression is that now, non-linear patterns such as the one mentioned with the `qtr` variable can be accounted for. However, unrestricted depth means the tree will keep branching until the data is essentially "pure", making it prone to overfitting.

In [11]:
clf_tree = DecisionTreeClassifier(random_state=42)
clf_tree.fit(X_train, y_train)

y_pred_tree = clf_tree.predict(X_test)

acc_tree = accuracy_score(y_test, y_pred_tree)
print(f"Overall accuracy: {acc_tree}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_tree)}")

pass_prob_tree = clf_tree.predict_proba(X_test)[:, 0]
auc_tree = roc_auc_score(y_test == 'pass', pass_prob_tree)

pred_dist_tree = pd.Series(y_pred_tree).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_tree.to_string()}\n")

print(f"AUC: {auc_tree}\n")

print(f"Training accuracy: {accuracy_score(y_train, clf_tree.predict(X_train))}")

Overall accuracy: 0.598562091503268

Classification Report:

              precision    recall  f1-score   support

        pass       0.63      0.65      0.64      4243
         run       0.55      0.53      0.54      3407

    accuracy                           0.60      7650
   macro avg       0.59      0.59      0.59      7650
weighted avg       0.60      0.60      0.60      7650

Predicted play type distribution:
pass    0.56915
run     0.43085

AUC: 0.5921052240188971

Training accuracy: 1.0


The single decision tree with unrestricted depth performs poorly. Its overall accuracy of 59.9% is only 4.4 percentage points above the 55.5% baseline, and well below the logistic model (65.5%). Both precision (0.63 vs 0.55) and recall (0.65 vs 0.53) are higher for 'pass', with the gap somewhat larger for recall. The model predicts pass on 56.9% of plays, only slightly above the true 55.5%, so unlike the logistic model it shows little bias towards the majority class. The AUC of 0.59 is only marginally above the 0.5 expected from random ranking. The reason for this weak performance is overfitting. The tree achieves a training accuracy of 100%, compared with 59.9% on the test set, because with no depth limit it keeps splitting until its leaves are completely pure. It therefore memorises noise in the training weeks, which does not generalise to unseen games.

### Model 3: Single Decision Tree (restricted depth)

Next, I again test a single decision tree, however this time with restricted depth. I restrict the depth to 5; this allows the model to capture non-linear relationships and interactions between several game-state variables, while preventing the tree from growing too deep and overfitting the data. This max depth was chosen heuristically rather than optimised by repeatedly running the model.

In [12]:
clf_tree_shallow = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_tree_shallow.fit(X_train, y_train)

y_pred_tree_shallow = clf_tree_shallow.predict(X_test)

acc_tree_shallow = accuracy_score(y_test, y_pred_tree_shallow)
print(f"Overall accuracy: {acc_tree_shallow}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_tree_shallow)}")

pass_prob_tree_shallow = clf_tree_shallow.predict_proba(X_test)[:, 0]
auc_tree_shallow = roc_auc_score(y_test == 'pass', pass_prob_tree_shallow)

pred_dist_tree_shallow = pd.Series(y_pred_tree_shallow).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_tree_shallow.to_string()}\n")

print(f"AUC: {auc_tree_shallow}\n")

print(f"Training accuracy: {accuracy_score(y_train, clf_tree_shallow.predict(X_train))}")

Overall accuracy: 0.6605228758169934

Classification Report:

              precision    recall  f1-score   support

        pass       0.70      0.68      0.69      4243
         run       0.62      0.63      0.62      3407

    accuracy                           0.66      7650
   macro avg       0.66      0.66      0.66      7650
weighted avg       0.66      0.66      0.66      7650

Predicted play type distribution:
pass    0.543007
run     0.456993

AUC: 0.7103706645473015

Training accuracy: 0.6652906776745786


The single decision tree with restricted depth is considerably more accurate than the unrestricted tree (66.1% vs 59.9%) and well above the 55.5% baseline, by 10.6 percentage points. Both precision (0.70 vs 0.62) and recall (0.68 vs 0.63) are higher for 'pass', with the gap larger for precision. The model predicts pass on 54.3% of plays, slightly below the true 55.5%, so it marginally overpredicts runs. This makes it the most balanced model so far, unlike the logistic model, which leaned towards passing. An AUC of 0.710 indicates a moderate ability to separate the two play types, but this is almost identical to the logistic model (0.706), and its accuracy is only 0.6 percentage points higher, suggesting that allowing non-linear relationships adds little at this depth. Crucially, the training accuracy of 66.5% is nearly identical to the test accuracy of 66.1%, in stark contrast to the unrestricted tree (100% vs 59.9%). This shows that limiting depth largely removes the overfitting problem.

### Model 4: Random Forest

The final model I test is a random forest. This model works by creating a large number of individual trees, then running all of them for a given play and predicting whatever the majority of trees predict. Each tree uses bootstrap row selection, meaning it is trained on approximately 63% of the total training set. Then, for each node, it restricts itself to only 3 random features out of the 9 total. Each individual tree is also capped at a depth of 10, chosen heuristically in the same way as the depth-5 cap used for the single restricted tree. Thus, each tree is different, with the idea being that their average will then lead to variance-reduction and give more accurate overall predictions.

In [13]:
clf_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
clf_rf.fit(X_train, y_train)

y_pred_rf = clf_rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Overall accuracy: {acc_rf}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_rf)}")

pass_prob_rf = clf_rf.predict_proba(X_test)[:, 0]
auc_rf = roc_auc_score(y_test == 'pass', pass_prob_rf)

pred_dist_rf = pd.Series(y_pred_rf).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_rf.to_string()}\n")

print(f"AUC: {auc_rf}")

Overall accuracy: 0.692156862745098

Classification Report:

              precision    recall  f1-score   support

        pass       0.72      0.72      0.72      4243
         run       0.65      0.66      0.66      3407

    accuracy                           0.69      7650
   macro avg       0.69      0.69      0.69      7650
weighted avg       0.69      0.69      0.69      7650

Predicted play type distribution:
pass    0.549281
run     0.450719

AUC: 0.75843121089443


The random forest achieves the highest accuracy so far (69.2%), 13.8 percentage points above the 55.5% baseline and 3.2 above the next best model, the restricted decision tree (66.1%). Both precision (0.72 vs 0.65) and recall (0.72 vs 0.66) are higher for 'pass', by a similar margin. The model predicts pass on 54.9% of plays, very close to the true 55.5%, making it the most balanced model so far. Its recall gap between the two classes is also much smaller than for the logistic model (0.72 vs 0.66, compared with 0.75 vs 0.54). Its AUC of 0.758 is again the highest, 0.048 above the restricted tree (0.710), indicating the best separation of pass and run plays. This suggests that combining many deeper trees captures more of the non-linear structure in the data than a single depth-5 tree, while averaging across trees reduces the variance that hurt the unrestricted tree.

### Part 2: Adding "pre-snap" features

Now, I add the two pre-snap features, `no_huddle` and `shotgun`, to the feature set and rerun all models to see if there is improvement in predictability. 

In [14]:
# Adding shotgun and no huddle features
X_train_presnap = train[['ydstogo', 'down', 'yardline_100', 
           'score_differential', 'game_seconds_remaining', 
           'home_timeouts_remaining', 'away_timeouts_remaining', 'qtr', 'rush_tendency', 
           'no_huddle', 'shotgun']]

X_test_presnap = test[['ydstogo', 'down', 'yardline_100', 
           'score_differential', 'game_seconds_remaining', 
           'home_timeouts_remaining', 'away_timeouts_remaining', 'qtr', 'rush_tendency',
           'no_huddle', 'shotgun']]

### Model 1: Logistic Regression

In [15]:
clf_log_presnap = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf_log_presnap.fit(X_train_presnap, y_train)

y_pred_log_presnap = clf_log_presnap.predict(X_test_presnap)

acc_log_presnap = accuracy_score(y_test, y_pred_log_presnap)
print(f"Overall accuracy: {acc_log_presnap}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_log_presnap)}")

pass_prob_log_presnap = clf_log_presnap.predict_proba(X_test_presnap)[:, 0]
auc_log_presnap = roc_auc_score(y_test == 'pass', pass_prob_log_presnap)

pred_dist_log_presnap = pd.Series(y_pred_log_presnap).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_log_presnap.to_string()}\n")

print(f"AUC: {auc_log_presnap}")

Overall accuracy: 0.7163398692810458

Classification Report:

              precision    recall  f1-score   support

        pass       0.71      0.81      0.76      4243
         run       0.72      0.59      0.65      3407

    accuracy                           0.72      7650
   macro avg       0.72      0.70      0.71      7650
weighted avg       0.72      0.72      0.71      7650

Predicted play type distribution:
pass    0.632549
run     0.367451

AUC: 0.7666961056249624


After adding the 'pre-snap' terms, we see a large improvement in accuracy for the logistic model (71.6% compared to 65.5%). The model now has high precision for both 'pass' and 'run', with the precision number being balanced for the two (71% and 72%). The gap in recall (81% to 59%) is slightly wider than in the first logistic model (75% to 54%), but only marginally. Both class recalls improved, so the imbalance stayed about the same rather than growing. The model predicts 'pass' 63.3% of the time, so again we see an overprediction of passing plays compared to the true number (55.5%), slightly larger than the 61.8% seen without pre-snap features. An AUC of 0.767 is an improvement on the logistic model with no pre-snap features (0.706), and now suggests a fairly strong ability to differentiate pass and run plays. Overall, we see that for the logistic model, adding 'pre-snap' features can yield a substantial improvement, yet doesn't solve the imbalance in overpredicting pass plays.

### Model 2: Single Decision Tree (unrestricted depth)

In [16]:
clf_tree_presnap = DecisionTreeClassifier(random_state=42)
clf_tree_presnap.fit(X_train_presnap, y_train)

y_pred_tree_presnap = clf_tree_presnap.predict(X_test_presnap)

acc_tree_presnap = accuracy_score(y_test, y_pred_tree_presnap)
print(f"Overall accuracy: {acc_tree_presnap}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_tree_presnap)}")

pass_prob_tree_presnap = clf_tree_presnap.predict_proba(X_test_presnap)[:, 0]
auc_tree_presnap = roc_auc_score(y_test == 'pass', pass_prob_tree_presnap)

pred_dist_tree_presnap = pd.Series(y_pred_tree_presnap).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_tree_presnap.to_string()}\n")

print(f"AUC: {auc_tree_presnap}")

Overall accuracy: 0.6205228758169935

Classification Report:

              precision    recall  f1-score   support

        pass       0.66      0.67      0.66      4243
         run       0.58      0.56      0.57      3407

    accuracy                           0.62      7650
   macro avg       0.62      0.61      0.62      7650
weighted avg       0.62      0.62      0.62      7650

Predicted play type distribution:
pass    0.563922
run     0.436078

AUC: 0.6149097520797908


Adding 'pre-snap' features to the unrestricted depth decision tree slightly improves accuracy (62.1% compared to 59.9%). The precision gap stays at 0.08 in favour of 'pass', while the recall gap decreases slightly (from 0.12 to 0.11). The predicted play distribution is near identical to the first unrestricted decision tree model, with 'pass' predicted 56.4% of the time, a minor decrease from 56.9% before, but still a marginal overprediction of 'pass' plays. An AUC of 0.615 is again slightly better than before (0.592), with the model remaining subpar at differentiating pass and run plays. Overall, adding 'pre-snap' terms improves the model only slightly, but the overfitting problem persists and greatly limits the model.

### Model 3: Single Decision Tree (restricted depth)

In [17]:
clf_tree_shallow_presnap = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_tree_shallow_presnap.fit(X_train_presnap, y_train)

y_pred_tree_shallow_presnap = clf_tree_shallow_presnap.predict(X_test_presnap)

acc_tree_shallow_presnap = accuracy_score(y_test, y_pred_tree_shallow_presnap)
print(f"Overall accuracy: {acc_tree_shallow_presnap}\n")
print(f"Training accuracy: {accuracy_score(y_train, clf_tree_shallow_presnap.predict(X_train_presnap))}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_tree_shallow_presnap)}")

pass_prob_tree_shallow_presnap = clf_tree_shallow_presnap.predict_proba(X_test_presnap)[:, 0]
auc_tree_shallow_presnap = roc_auc_score(y_test == 'pass', pass_prob_tree_shallow_presnap)

pred_dist_tree_shallow_presnap = pd.Series(y_pred_tree_shallow_presnap).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_tree_shallow_presnap.to_string()}\n")

print(f"AUC: {auc_tree_shallow_presnap}")

Overall accuracy: 0.7163398692810458

Training accuracy: 0.7138372893016856

Classification Report:

              precision    recall  f1-score   support

        pass       0.72      0.80      0.76      4243
         run       0.71      0.62      0.66      3407

    accuracy                           0.72      7650
   macro avg       0.72      0.71      0.71      7650
weighted avg       0.72      0.72      0.71      7650

Predicted play type distribution:
pass    0.613725
run     0.386275

AUC: 0.768003080541296


Adding 'pre-snap' features to the decision tree with restricted depth substantially improves accuracy (71.6% compared to 66.1%, a gain of 5.6 percentage points). Precision is only marginally better for 'pass' plays (72% vs 71%), whereas recall is now far better for 'pass' (80% vs 62%). This split wasn't there for the restricted tree without 'pre-snap' features, where the precision and recall gaps were closer in size (8 points and 5 points respectively, both favouring 'pass'). Notably, adding 'pre-snap' features now makes the model overpredict 'pass', predicting it 61.4% of the time, up from a slight underprediction at 54.3%, compared to the true 55.5% value. The AUC improves from 0.710 to 0.768, a gain of 0.058, so adding 'pre-snap' terms clearly improves the model's ability to differentiate pass plays from run plays. Overall, adding 'pre-snap' terms improves the model substantially in both accuracy and AUC, yet imbalances it in favour of predicting 'pass' plays.

### Model 4: Random Forest

In [18]:
clf_rf_presnap = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
clf_rf_presnap.fit(X_train_presnap, y_train)

y_pred_rf_presnap = clf_rf_presnap.predict(X_test_presnap)

acc_rf_presnap = accuracy_score(y_test, y_pred_rf_presnap)
print(f"Overall accuracy: {acc_rf_presnap}\n")
print(f"Classification Report:\n\n{classification_report(y_test, y_pred_rf_presnap)}")

pass_prob_rf_presnap = clf_rf_presnap.predict_proba(X_test_presnap)[:, 0]
auc_rf_presnap = roc_auc_score(y_test == 'pass', pass_prob_rf_presnap)

pred_dist_rf_presnap = pd.Series(y_pred_rf_presnap).value_counts(normalize=True)
print(f"Predicted play type distribution:\n{pred_dist_rf_presnap.to_string()}\n")

print(f"AUC: {auc_rf_presnap}")

Overall accuracy: 0.725359477124183

Classification Report:

              precision    recall  f1-score   support

        pass       0.73      0.79      0.76      4243
         run       0.71      0.64      0.67      3407

    accuracy                           0.73      7650
   macro avg       0.72      0.72      0.72      7650
weighted avg       0.72      0.73      0.72      7650

Predicted play type distribution:
pass    0.600784
run     0.399216

AUC: 0.7900063095340789


Adding 'pre-snap' features to the random forest model improves overall accuracy from 69.2% to 72.5%, a gain of 3.3 percentage points. Both precision and recall are better for pass, with this gap being much larger for recall (15 points) than for precision (2 points). This differs from the random forest without 'pre-snap' features, where 'pass' performed better on precision and recall by similar amounts (7 points and 6 points). The model now also overpredicts 'pass' at 60.1%, compared to a slight underprediction of 54.9% before, relative to the true 55.5%. The AUC likewise improves from 0.758 to 0.790, a gain of 0.032, showing the model now has a strong ability to differentiate 'pass' and 'run' plays. Overall, adding 'pre-snap' features to the random forest improves its accuracy and AUC, at the cost of now overpredicting 'pass' plays.

## Model Comparison

To get a better overall picture, I compare all 8 models to each other across the key metrics.

### Accuracy

In [19]:
# Compare accuracy across all 4
accuracy_table = pd.DataFrame({
    "Without presnap info": [
        acc_log,
        acc_tree,
        acc_tree_shallow,
        acc_rf
    ],
    "With presnap info": [
        acc_log_presnap,
        acc_tree_presnap,
        acc_tree_shallow_presnap,
        acc_rf_presnap
    ]
}, index=[
    "Logistic Regression",
    "Decision Tree",
    "Shallow Decision Tree",
    "Random Forest"
])

accuracy_table['Improvement'] = accuracy_table['With presnap info'] - accuracy_table['Without presnap info']

print(accuracy_table)

                       Without presnap info  With presnap info  Improvement
Logistic Regression                0.654641           0.716340     0.061699
Decision Tree                      0.598562           0.620523     0.021961
Shallow Decision Tree              0.660523           0.716340     0.055817
Random Forest                      0.692157           0.725359     0.033203


Overall, the random forest model outperforms all other models on accuracy both with and without 'pre-snap' features included in the feature list. We see that adding 'pre-snap' features to the feature list improves accuracy in every model, implying that these two added features have some relevant predictive power even on top of 'game-context' features. This improvement is largest for logistic regression (6.1 percentage points), followed closely by the shallow decision tree (5.6 percentage points). The random forest improves by only 3.3 percentage points, while the unrestricted decision tree improves the least, by just 2.2 percentage points. After adding 'pre-snap' context, the logistic regression and shallow decision tree models become nearly as accurate as the random forest (71.6% vs 71.6% vs 72.5%), whereas with only 'game-context', they clearly lag behind the random forest (65.5% vs 66.1% vs 69.2%). Thus, 'pre-snap' context appears to carry more additional signal for logistic regression and a single restricted decision tree than for a random forest.

One possible reason for this is diminishing returns: the feature set as a whole may be approaching a predictive ceiling, with all four models converging towards a similar accuracy once pre-snap features are added, regardless of model type. Another is that the random forest may already have captured much of the same signal indirectly, by combining several 'game-context' features (such as down, distance and score) in ways a linear model or a shallow tree cannot. Since shotgun and no-huddle usage is itself driven by situational factors like down and distance, a flexible model may partly reconstruct this signal without ever seeing the features directly, meaning the pre-snap features add less new information for the random forest than for the simpler models.

### AUC

In [20]:
# Compare AUC across all 4
auc_table = pd.DataFrame({
    "Without presnap info": [
        auc_log,
        auc_tree,
        auc_tree_shallow,
        auc_rf
    ],
    "With presnap info": [
        auc_log_presnap,
        auc_tree_presnap,
        auc_tree_shallow_presnap,
        auc_rf_presnap
    ]
}, index=[
    "Logistic Regression",
    "Decision Tree",
    "Shallow Decision Tree",
    "Random Forest"
])

auc_table['Improvement'] = auc_table['With presnap info'] - auc_table['Without presnap info']

print(auc_table)

                       Without presnap info  With presnap info  Improvement
Logistic Regression                0.706410           0.766696     0.060286
Decision Tree                      0.592105           0.614910     0.022805
Shallow Decision Tree              0.710371           0.768003     0.057632
Random Forest                      0.758431           0.790006     0.031575


Overall, the random forest outperforms all other models on AUC, both with and without 'pre-snap' features. The shallow decision tree and logistic regression models are fairly evenly matched, while the unrestricted decision tree is the worst by far. Adding 'pre-snap' features sees all models improve in AUC, with the improvement again being largest for logistic regression (6.0 percentage points), followed closely by the shallow decision tree (5.8 percentage points). The random forest improves by only about half as much (3.2 percentage points), while the unrestricted decision tree sees the smallest improvement (2.3 percentage points). This indicates that adding 'pre-snap' features helps the logistic regression and shallow decision tree models more in differentiating 'pass' and 'run' plays than it does to the random forest. However, unlike in accuracy, the random forest still maintains a substantial lead in AUC (0.790 v 0.768 v 0.767). 

The smaller improvement for random forest is again likely because it can already incorporate some of that signal of the 'pre-snap' features into its model via interactions of other features. However, its AUC is still 0.022 higher than the next best model, likely because averaging 200 trees gives it much finer-grained probabilities than a single shallow tree (capped at 32 possible values) or a linear model. This lets it rank borderline plays more accurately, which AUC rewards but accuracy, being threshold-based, doesn't fully capture.

### Predicted play-type distribution

In [21]:
pred_pass_table = pd.DataFrame({
    "Without presnap info": [
        pred_dist_log['pass'],
        pred_dist_tree['pass'],
        pred_dist_tree_shallow['pass'],
        pred_dist_rf['pass']
    ],
    "With presnap info": [
        pred_dist_log_presnap['pass'],
        pred_dist_tree_presnap['pass'],
        pred_dist_tree_shallow_presnap['pass'],
        pred_dist_rf_presnap['pass']
    ]
}, index=[
    "Logistic Regression",
    "Decision Tree",
    "Shallow Decision Tree",
    "Random Forest"
])

pred_pass_table_diff = pred_pass_table - 0.555

pred_pass_table_diff['Difference'] = pred_pass_table_diff['With presnap info'] - pred_pass_table_diff['Without presnap info']

print(pred_pass_table_diff)

                       Without presnap info  With presnap info  Difference
Logistic Regression                0.063170           0.077549    0.014379
Decision Tree                      0.014150           0.008922   -0.005229
Shallow Decision Tree             -0.011993           0.058725    0.070719
Random Forest                     -0.005719           0.045784    0.051503


Without 'pre-snap' features, we see that the random forest is closest to a balanced prediction. The shallow decision tree slightly overpredicts 'run' plays, while the unrestricted decision tree slightly overpredicts 'pass' plays. The logistic regression model then substantially overpredicts 'pass' plays (by 6.3 percentage points). However, when 'pre-snap' information is added, we see most models shift in favour of overpredicting 'pass' plays. This is the case for logistic regression (6.3 to 7.7 percentage points), which sees a slight increase, and especially the shallow decision tree (-1.2 to +5.9 percentage points) and random forest (-0.6 to +4.6 percentage points) models, which go from slightly overpredicting 'run' to substantially overpredicting 'pass'.

This is likely due to the `shotgun` feature, one of the two 'pre-snap' features added. When an offense lines up in shotgun formation, it is a strong signal that it will pass the ball. However, not lining up in shotgun can still often lead to a pass as well, since shotgun is far from the only situation in which teams pass. This asymmetry means the models can now more confidently flag likely passing plays, while running plays remain comparatively harder to pin down, causing them to favour overpredicting 'pass'.

## Conclusion 

Aim by aim, what results and conclusions has this project yielded?

1) The highest overall predictive accuracy was 72.5% (random forest with 'pre-snap' info).

2) 'Pre-snap' features have additional predictive power even on top of 'game-context' features. However, these specific 'pre-snap' features (`shotgun` and `no_huddle`) seem to be better indicators of 'pass' plays, causing models to overpredict 'pass' plays.

3) This project is a good demonstration of many key properties of the selected models. Here are the major takeaways:
    - An unrestricted single tree will overfit on training data and as a result perform badly at predictivity, only slightly outperforming the 55.5% benchmark.
    - A random forest's ability to capture many feature interactions means it can partially pick up the signal in 'pre-snap' variables even before they're added, unlike logistic regression and the shallow decision tree. This is why adding 'pre-snap' features improves the random forest's accuracy and AUC by less than it does for the other two models.
    - Even with 'pre-snap' features, the random forest maintains a larger lead over the other models in AUC than in accuracy. This reflects how finely each model can rank plays: the shallow decision tree is capped at 32 distinct probability values (one per leaf), so many plays get tied, identical probabilities, while logistic regression produces continuous probabilities but is constrained to a linear decision boundary, which can still misorder plays with non-linear relationships. The random forest, by averaging across 200 trees, produces much finer-grained probabilities that avoid both limitations, letting it rank plays more accurately even as its accuracy converges with the other models.